# Sua ket qua — KHONG train lai, chi danh gia lai tu checkpoint da luu

| JOB | Viec | ~phut |
|---|---|---|
| `fmi_mia` | Eval lai E30 cua Forget-MI o CA 4 cau hinh, dung tap member 512 | ~40 |
| `p3_iu_logits` | Chan doan AUC=NaN cua P3 IU E30: logit tran FP16 hay sup do that | ~15 |

## Vi sao phai lam

**`fmi_mia`** — `_final_evaluation` cua Forget-MI bo qua `eval_max_retain=512` va truyen
THANG toan bo retain (~5410) lam tap member, trong khi selector va P3 deu dung ban lay
mau 512. Hau qua: cung mot checkpoint E30 cho ra MIA khac nhau, va `MIA_paper` bao hoa
= 1.000 o ca 3/6/10% do mat can bang 5410 : 398. Cot MIA/MIA_paper hang E30 hien KHONG
so duoc giua hai phuong phap.
`forgetmi_eval_only.py` dung `final_evaluation` cua adv_common (CO lay mau 512) nen chay
lai bang script nay la du — khong sua code, khong train lai.

**`p3_iu_logits`** — P3 IU E30 cho AUC = NaN. Mo hinh doan te thuong cho AUC ~0.5 chu
khong thanh NaN, nen phai tach ba kha nang: (A) logit khong huu han ngay o FP32 → sup do
that; (B) huu han ca hai ma AUC van NaN → loi ham metric; (C) huu han o FP32 nhung tran
duoi autocast FP16 luc eval → artefact precision, KHONG phai sup do. Ket luan trong khoa
luan khac han giua A va C.

## Chuan bi INPUT (quan trong)

Add Input → **Your Work → Notebook Output** cua cac session cu de lay file `.pt`:

- `fmi_mia`: 4 file `last.pt` (~450 MB/file) cua fmi 3%/6%/10%/IU
- `p3_iu_logits`: `latest.pt` (~6 MB) cua p3_iu

Cell 2 se TU DO tim cac file nay trong /kaggle/input. Neu khong thay se bao ro thieu cai nao.


In [ ]:
# Cell 1: setup + CHOT CHAN code da push
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/adv_common.py'),'push code truoc + re-import notebook'
_adv=open('training/adv_common.py').read()
assert 'ce_selector' in _adv and 'checkpoint_selection_' in _adv, \
    '❌ adv_common CHUA co hook CE-selector -> chay `git push` code MOI roi moi Save Version!'
assert 'OnlineCESelector' in open('training/ce_selector_pilot.py').read(), '❌ git push code moi truoc!'
assert os.path.exists('training/forgetmi_p3_cand.py'), '❌ chua push forgetmi_p3_cand.py!'
print('✅ Code CE-selector da co (hook + OnlineCESelector).')
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CHON JOB + tim checkpoint
import glob, os
JOB = 'fmi_mia'      # 'fmi_mia' | 'p3_iu_logits'
SEED = 42
assert JOB in ('fmi_mia','p3_iu_logits')

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)
def first_existing(root, rels):
    for r in rels:
        p=os.path.join(root,r)
        if os.path.exists(p): return p
    return None

# ---- tim checkpoint trong MOI input da gan ----
def find_ckpt(*pats):
    hits=[]
    for pat in pats:
        hits += glob.glob(f'/kaggle/input/**/{pat}', recursive=True)
    return sorted(set(hits), key=len)

print('=== checkpoint tim thay ===')
CK={}
if JOB=='fmi_mia':
    # last.pt cua Forget-MI: .../<JOB>_s42/<0.17_UKR_...>/checkpoints/last.pt
    for tag, key in [('mimic3per','fmi_mimic3per_s42'), ('mimic6per','fmi_m6_s42'),
                     ('mimic10per','fmi_m10_s42'), ('iu3per','fmi_iu_s42')]:
        h=[p for p in find_ckpt('checkpoints/last.pt') if key in p]
        CK[tag]=h[0] if h else None
        print(f'  {tag:11} {key:18} -> {CK[tag] or "KHONG THAY"}')
    missing=[k for k,v in CK.items() if not v]
    assert not missing, f'Thieu last.pt cho: {missing}. Add Input notebook output tuong ung.'
else:
    h=[p for p in find_ckpt('checkpoints/latest.pt') if 'p3_iu' in p]
    CK['p3_iu']=h[0] if h else None
    print('  p3_iu latest.pt ->', CK['p3_iu'] or 'KHONG THAY')
    assert CK['p3_iu'], 'Thieu latest.pt cua p3_iu. Add Input notebook output cua run p3_iu.'

# ---- path du lieu ----
DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
DATA_IU=fd('forget-mi-data-iu'); MOD_IU=fd('forget-mi-models-iu'); MODRE_IU=fd('forget-mi-models-iu-re')
RAD=fd('chest-xrays-indiana-university')

def mimic_paths(pct):
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gh=[b for b in bins(MOD) if f'model_retrained_{pct}per' in b]
    return {'base_model_path':BASE,'bert_pretrained_dir':BASE,
            'retrained_model_path':os.path.dirname(gh[0]) if gh else BASE,
            'text_data_dir':os.path.join(DATA,'data','metadata'),
            'img_data_dir':os.path.join(DATA,'data','img_data'),
            'data_split_path':'./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv',
            'forget_set_path':f'./data_splits/forget_set_{pct}per.csv'}

def iu_paths():
    ogb=[b for b in bins(MOD_IU) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD_IU)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE_IU) if MODRE_IU else []) or [b for b in bins(MOD_IU) if 'retrain' in b.lower()]
    tsv=glob.glob(os.path.join(DATA_IU,'**','all_data.tsv'),recursive=True) or glob.glob('/kaggle/input/**/all_data.tsv',recursive=True)
    sp=glob.glob(os.path.join(DATA_IU,'**','iu-split.csv'),recursive=True) or glob.glob('/kaggle/input/**/iu-split.csv',recursive=True)
    fg=glob.glob(os.path.join(DATA_IU,'**','forget_set_3per_iu.csv'),recursive=True) or glob.glob('/kaggle/input/**/forget_set_3per_iu.csv',recursive=True)
    IMG=first_existing(DATA_IU,['data/img_data','img_data']) or (first_existing(RAD,['images/images_normalized','images']) if RAD else None) or RAD
    return {'base_model_path':BASE,'bert_pretrained_dir':BASE,
            'retrained_model_path':os.path.dirname(reb[0]) if reb else BASE,
            'text_data_dir':os.path.dirname(tsv[0]) if tsv else first_existing(DATA_IU,['data/metadata','metadata']),
            'img_data_dir':IMG,'data_split_path':sp[0],'forget_set_path':fg[0]}

RESULTS=f'/kaggle/working/results_{JOB}.csv'
print('\nJOB',JOB,'| RESULTS',RESULTS)


In [ ]:
# Cell 3: CHAY
import os, subprocess, time
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled',
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
t0=time.time()

if JOB=='fmi_mia':
    # Eval lai E30 cua Forget-MI bang final_evaluation cua adv_common (member = 512).
    # Dung CUNG config voi P3 o tung bo du lieu de giao thuc danh gia trung khop.
    JOBS=[('mimic3per', 3,'config_advanced_kaggle.yaml'),
          ('mimic6per', 6,'config_advanced_kaggle.yaml'),
          ('mimic10per',10,'config_advanced_kaggle.yaml'),
          ('iu3per',    3,'config_loku_iu_kaggle.yaml')]
    for tag,pct,cfgf in JOBS:
        ovr = (iu_paths() if tag=='iu3per' else mimic_paths(pct))
        ovr['results_csv_path']=RESULTS
        ovr['output_dir']=f'/kaggle/working/fix_{tag}'
        cmd=['python','training/forgetmi_eval_only.py','--config',cfgf,'--seed',str(SEED),
             '--label',f'fmi_e30_{tag}','--model_type','state_dict','--model_path',CK[tag],
             '--method','forgetmi','--override',','.join(f'{k}={v}' for k,v in ovr.items())]
        print('='*72+f'\nEVAL LAI  {tag}\n  {CK[tag]}\n'+'='*72)
        try: subprocess.run(cmd,env=env,check=True)
        except subprocess.CalledProcessError as e: print('FAIL',tag,'rc=',e.returncode)
else:
    ovr = iu_paths()
    ovr['output_dir']='/kaggle/working/diag_p3_iu'
    ovr['results_csv_path']='/kaggle/working/diag_p3_iu.csv'
    cmd=['python','tools/diagnose_logits.py','--config','config_loku_iu_kaggle.yaml',
         '--seed',str(SEED),'--ckpt',CK['p3_iu'],
         '--override',','.join(f'{k}={v}' for k,v in ovr.items())]
    print('='*72+f'\nCHAN DOAN LOGIT  p3_iu E30\n  {CK["p3_iu"]}\n'+'='*72)
    try: subprocess.run(cmd,env=env,check=True)
    except subprocess.CalledProcessError as e: print('FAIL rc=',e.returncode)

print(f'\nXONG {(time.time()-t0)/60:.1f} phut')


In [ ]:
# Cell 4: doc ket qua (chi co y nghia voi JOB='fmi_mia')
import os, pandas as pd
pd.set_option('display.width',220)
if JOB=='fmi_mia' and os.path.exists(RESULTS):
    d=pd.read_csv(RESULTS)
    cols=[c for c in ['id','method','checkpoint_kind','MIA','MIA_paper','Forget_AUC',
                      'Forget_Macro_F1','Test_AUC','Test_Macro_F1','forget_ce','test_ce']
          if c in d.columns]
    print('===== MIA E30 tinh LAI (member = 512, cung duong voi P3 va voi S2) =====')
    print(d[cols].to_string(index=False))
    print('''
So sanh voi so CU trong bang (member = toan bo retain ~5410):
  MIMIC 3%   MIA 0.567  MIA_paper 1.000
  MIMIC 6%   MIA 0.544  MIA_paper 1.000
  MIMIC 10%  MIA 0.559  MIA_paper 1.000
  IU 3%      MIA 0.602  MIA_paper 0.667
Neu so moi khac ro -> dung so MOI cho hang E30 trong Chuong 4.''')
elif JOB=='p3_iu_logits':
    print('Doc thang phan ">>> KET LUAN" trong log Cell 3 (truong hop A / B / C).')
else:
    print('chua co', RESULTS)
print('\nTAI VE: results_fmi_mia.csv (va toan bo log Cell 3 neu chay chan doan)')
